# Module 06 — AI Workbook Companion (GPU calorimeter transport)

A lightweight companion to
[gpu_dd4hep_tilecal.ipynb](../gpu_dd4hep_tilecal.ipynb) and
[gpu_geant4_dd4hep_cuda_notebook.md](../gpu_geant4_dd4hep_cuda_notebook.md). It
does **not** replace them — it adds the supervision layer from
[Module 11](../../11/README.md) to Module 06's topic: **accumulating energy
deposits into calorimeter cells on the GPU** (the pattern behind Celeritas /
offloaded Geant4 transport).

New here? Read [Module 11's README](../../11/README.md) first. The relevant
failure mode here is a **scatter race**: many parallel steps deposit energy into
the same detector cell, and without an atomic add those deposits clobber each
other. It is the same bug as a GPU histogram without `atomicAdd`, in physics
clothing.

## The loop, applied to energy deposition

Your task: take a list of energy deposits, each tagged with the calorimeter cell
it lands in, and sum them per cell — the way a GPU transport code fills a hit
map. You may have an AI write it. Your job is everything around it.

1. **SPECIFY** — Contract: deposit arrays (cell id, energy), number of cells,
   what "correct" means (total energy conserved; each cell is the exact sum of
   its deposits), and the metric.
2. **GENERATE** — Ask the AI for the per-cell accumulation. Keep it separate.
3. **PREDICT** — *Before running:* the risk is `cells[cell_id] += edep` with
   repeated cell ids, which does not accumulate. Do you see it?
4. **VERIFY** — Use [verify_calorimeter.py](./verify_calorimeter.py): a reference
   (`np.add.at` / weighted `np.bincount`) with total-energy and per-cell PASS/FAIL.
5. **DIAGNOSE** — Show that total deposited energy is not conserved in the buggy
   version, and give the GPU fix (`cuda.atomic.add`).

## The adversarial exercise

[adversarial_calorimeter_buggy.py](./adversarial_calorimeter_buggy.py) is a
hit-map accumulation "an AI wrote for you." It runs and produces a plausible
energy map, but total energy is **not conserved** because repeated cell writes
overwrite instead of accumulate — exactly a GPU deposition without atomics.

Your job:

1. Predict the failure and its signature (energy non-conservation) *before* running.
2. Run it and compare total energy in vs total energy in the map.
3. State the exact bug and the fix.
4. Prove the fix: total energy conserved and every cell matches the reference.

Could an AI catch this? Sometimes — but an energy map that "looks like a shower"
passes a casual review while silently violating energy conservation. Checking
`sum(map) == sum(deposits)` and comparing per cell catches it. That is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why does `cells[cell_id] += edep` lose energy when cell ids repeat? How is that
  identical to a GPU deposition without `atomicAdd`?
- Energy conservation is a physics invariant. What made it a *better* correctness
  test here than "the shower profile looks right"?
- On a GPU, atomic adds into hot cells can serialize. What is the next
  optimization (per-block privatized maps), and how would you measure its effect?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Run the problem program [adversarial_calorimeter_buggy.py](./adversarial_calorimeter_buggy.py). Read it first and predict what will happen before you run this cell.

In [ ]:
!python adversarial_calorimeter_buggy.py

## Step 2 - Run your verification harness

[verify_calorimeter.py](./verify_calorimeter.py) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the function under test. Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!python verify_calorimeter.py

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.